# Synthetic Control — Airbnb Experiences Launch

**Business Problem:** Airbnb launches its "Experiences" product in Austin, TX. Leadership wants to know the causal effect on total booking revenue. Since only Austin received the launch (and other cities did not), we cannot run a standard A/B test or DiD with many treated units. Instead, we construct a **synthetic Austin** from a weighted combination of donor cities to estimate what Austin's revenue *would have been* without the launch.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## Step 1: Simulate Panel Data

**WHY:** We need a realistic dataset — 15 cities observed over 36 months. Austin receives the Experiences launch at month 24. Each city has a different baseline revenue level and slightly different trend, plus city-specific noise. The **true treatment effect is a +15% revenue lift** for Austin post-launch. This known ground truth lets us validate whether our synthetic control recovers the correct estimate.

In [ ]:
n_cities = 15
n_months = 36
treatment_month = 24
true_lift = 0.15

cities = [
    "Austin", "Denver", "Nashville", "Portland", "Raleigh",
    "Salt Lake City", "Charlotte", "Minneapolis", "Tampa",
    "Pittsburgh", "Columbus", "Indianapolis", "Kansas City",
    "Milwaukee", "Cincinnati"
]

baselines = {
    "Austin": 500, "Denver": 480, "Nashville": 520, "Portland": 460,
    "Raleigh": 440, "Salt Lake City": 420, "Charlotte": 470,
    "Minneapolis": 450, "Tampa": 510, "Pittsburgh": 430,
    "Columbus": 455, "Indianapolis": 435, "Kansas City": 445,
    "Milwaukee": 425, "Cincinnati": 465
}

trends = {
    "Austin": 8.0, "Denver": 7.5, "Nashville": 8.5, "Portland": 6.5,
    "Raleigh": 6.0, "Salt Lake City": 5.5, "Charlotte": 7.0,
    "Minneapolis": 6.8, "Tampa": 8.2, "Pittsburgh": 5.8,
    "Columbus": 6.5, "Indianapolis": 5.9, "Kansas City": 6.2,
    "Milwaukee": 5.4, "Cincinnati": 7.1
}

months = np.arange(1, n_months + 1)
common_shock = np.random.normal(0, 5, n_months)

records = []
for city in cities:
    base = baselines[city]
    trend = trends[city]
    noise = np.random.normal(0, 10, n_months)
    revenue = base + trend * months + common_shock + noise

    if city == "Austin":
        post_mask = months > treatment_month
        counterfactual = revenue.copy()
        revenue[post_mask] = revenue[post_mask] * (1 + true_lift)

    for i, m in enumerate(months):
        records.append({"city": city, "month": m, "revenue": revenue[i]})

df = pd.DataFrame(records)

print(f"Panel shape: {df.shape}")
print(f"Cities: {df['city'].nunique()}, Months: {df['month'].nunique()}")
print(f"Treatment month: {treatment_month}")
print(f"True lift: {true_lift:.0%}")
df.head(10)

## Step 2: Naive Comparison (Why It Fails)

The simplest approach is to compare Austin's post-treatment revenue to the average of all other cities. But this ignores that Austin may have a **different baseline level and trend** from the average donor. Any pre-existing gap gets conflated with the treatment effect.

In [ ]:
austin_df = df[df["city"] == "Austin"].set_index("month")["revenue"]
donor_avg = df[df["city"] != "Austin"].groupby("month")["revenue"].mean()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(austin_df.index, austin_df.values, label="Austin (actual)", linewidth=2)
ax.plot(donor_avg.index, donor_avg.values, label="Average of other cities", linewidth=2, linestyle="--")
ax.axvline(x=treatment_month, color="red", linestyle=":", linewidth=1.5, label="Experiences launch")
ax.set_xlabel("Month")
ax.set_ylabel("Booking Revenue ($K)")
ax.set_title("Naive Comparison: Austin vs. Average of Other Cities")
ax.legend()
plt.tight_layout()
plt.show()

naive_gap = austin_df[austin_df.index > treatment_month].mean() - donor_avg[donor_avg.index > treatment_month].mean()
print(f"Naive post-treatment gap: ${naive_gap:.1f}K")
print("⚠️  This conflates Austin's higher baseline/trend with the treatment effect.")

## Step 3: Why Synthetic Control, Not Other Methods?

| Method | Why it doesn't fit here |
|--------|------------------------|
| **Difference-in-Differences (DiD)** | Requires a group of treated units for reliable parallel-trends comparison. We have only **one** treated city (Austin). |
| **Interrupted Time Series (ITS)** | Only uses Austin's own pre-treatment trend to project the counterfactual. SC does better by leveraging **14 donor cities** to build a more robust counterfactual. |
| **Propensity Score Matching (PSM)** | Designed for **individual-level** treatment assignment. Here treatment is at the city level — there are no individual propensity scores to estimate. |

**Synthetic Control is ideal** because it constructs a data-driven counterfactual from a donor pool, works with a single treated unit, and provides transparent, interpretable weights.

## Step 4: Build the Synthetic Control

**WHY:** We find non-negative weights on the 14 donor cities so that the weighted combination of their pre-treatment revenue trajectories matches Austin's pre-treatment trajectory as closely as possible. The better the pre-treatment fit, the more credible the post-treatment gap is as a causal estimate.

Formally, we solve:

$$\min_{w} \sum_{t=1}^{T_0} \left( Y_{\text{Austin},t} - \sum_{j} w_j \, Y_{j,t} \right)^2 \quad \text{s.t.} \quad w_j \geq 0, \; \sum_j w_j = 1$$

In [ ]:
pivot = df.pivot(index="month", columns="city", values="revenue")
donor_cities = [c for c in cities if c != "Austin"]

pre_austin = pivot.loc[:treatment_month, "Austin"].values
pre_donors = pivot.loc[:treatment_month, donor_cities].values

def objective(w):
    synthetic = pre_donors @ w
    return np.sum((pre_austin - synthetic) ** 2)

n_donors = len(donor_cities)
constraints = [
    {"type": "eq", "fun": lambda w: np.sum(w) - 1}
]
bounds = [(0, 1)] * n_donors
w0 = np.ones(n_donors) / n_donors

result = minimize(objective, w0, method="SLSQP", bounds=bounds, constraints=constraints)
weights = result.x

print("Donor City Weights for Synthetic Austin:")
print("=" * 40)
for city, w in sorted(zip(donor_cities, weights), key=lambda x: -x[1]):
    if w > 0.001:
        print(f"  {city:20s}: {w:.4f}")
print(f"\nWeights sum: {weights.sum():.4f}")
print(f"Optimization success: {result.success}")

In [ ]:
all_donors = pivot[donor_cities].values
synthetic_austin = all_donors @ weights
actual_austin = pivot["Austin"].values

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(months, actual_austin, label="Austin (actual)", linewidth=2.5, color="#e74c3c")
ax.plot(months, synthetic_austin, label="Synthetic Austin", linewidth=2.5, linestyle="--", color="#2c3e50")
ax.axvline(x=treatment_month, color="gray", linestyle=":", linewidth=1.5, label="Experiences launch")
ax.fill_between(
    months[treatment_month:], actual_austin[treatment_month:], synthetic_austin[treatment_month:],
    alpha=0.2, color="#e74c3c", label="Treatment effect"
)
ax.set_xlabel("Month")
ax.set_ylabel("Booking Revenue ($K)")
ax.set_title("Synthetic Control: Austin vs. Synthetic Austin")
ax.legend()
plt.tight_layout()
plt.show()

pre_rmse = np.sqrt(np.mean((actual_austin[:treatment_month] - synthetic_austin[:treatment_month]) ** 2))
print(f"Pre-treatment RMSE: ${pre_rmse:.2f}K (lower is better)")

## Step 5: Estimate the Treatment Effect

The **gap** between actual Austin and synthetic Austin in the post-treatment period is our causal estimate of the Experiences launch effect. A positive gap means the launch increased revenue.

In [ ]:
gap = actual_austin - synthetic_austin
post_gap = gap[treatment_month:]
post_actual = actual_austin[treatment_month:]
post_synthetic = synthetic_austin[treatment_month:]

avg_gap = np.mean(post_gap)
pct_lift = np.mean(post_gap / post_synthetic) * 100

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(months, gap, linewidth=2, color="#2c3e50")
ax.axhline(y=0, color="gray", linestyle="-", linewidth=0.8)
ax.axvline(x=treatment_month, color="red", linestyle=":", linewidth=1.5, label="Experiences launch")
ax.fill_between(months[treatment_month:], 0, gap[treatment_month:], alpha=0.3, color="#e74c3c")
ax.set_xlabel("Month")
ax.set_ylabel("Gap: Actual − Synthetic ($K)")
ax.set_title("Treatment Effect: Gap Between Actual and Synthetic Austin")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Average post-treatment gap: ${avg_gap:.1f}K per month")
print(f"Estimated percentage lift: {pct_lift:.1f}%")
print(f"True lift (ground truth): {true_lift:.0%}")

## Step 6: Placebo Tests for Inference

**WHY:** With only one treated unit, we can't compute a traditional standard error. Instead, we use **placebo (permutation) tests**: apply the synthetic control method to *each donor city* as if it were the treated unit. If Austin's post-treatment gap is larger than all (or nearly all) placebo gaps, we conclude the effect is statistically significant.

The **p-value** = (rank of Austin's gap among all gaps) / (total number of units).

In [ ]:
def run_synthetic_control(treated_city, df, cities, treatment_month):
    """Run SC for a given treated city and return the gap series."""
    pivot = df.pivot(index="month", columns="city", values="revenue")
    donors = [c for c in cities if c != treated_city]

    pre_treated = pivot.loc[:treatment_month, treated_city].values
    pre_donor_mat = pivot.loc[:treatment_month, donors].values

    def obj(w):
        return np.sum((pre_treated - pre_donor_mat @ w) ** 2)

    n_d = len(donors)
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
    bds = [(0, 1)] * n_d
    w0 = np.ones(n_d) / n_d

    res = minimize(obj, w0, method="SLSQP", bounds=bds, constraints=cons)
    all_donor_mat = pivot[donors].values
    synth = all_donor_mat @ res.x
    actual = pivot[treated_city].values

    pre_rmse = np.sqrt(np.mean((actual[:treatment_month] - synth[:treatment_month]) ** 2))
    return actual - synth, pre_rmse

placebo_gaps = {}
placebo_pre_rmse = {}

for city in cities:
    gap_series, pre_rmse = run_synthetic_control(city, df, cities, treatment_month)
    placebo_gaps[city] = gap_series
    placebo_pre_rmse[city] = pre_rmse

fig, ax = plt.subplots(figsize=(12, 6))
for city in donor_cities:
    ax.plot(months, placebo_gaps[city], color="lightgray", linewidth=0.8, alpha=0.7)
ax.plot(months, placebo_gaps["Austin"], color="#e74c3c", linewidth=2.5, label="Austin")
ax.axhline(y=0, color="gray", linestyle="-", linewidth=0.8)
ax.axvline(x=treatment_month, color="gray", linestyle=":", linewidth=1.5)
ax.set_xlabel("Month")
ax.set_ylabel("Gap: Actual − Synthetic ($K)")
ax.set_title("Placebo Tests: Austin (red) vs. All Donor Cities (gray)")
ax.legend()
plt.tight_layout()
plt.show()

austin_post_rmspe = np.sqrt(np.mean(placebo_gaps["Austin"][treatment_month:] ** 2))
post_rmspes = {city: np.sqrt(np.mean(g[treatment_month:] ** 2)) for city, g in placebo_gaps.items()}

rank = sum(1 for v in post_rmspes.values() if v >= austin_post_rmspe)
p_value = rank / len(cities)

print(f"Austin post-treatment RMSPE: {austin_post_rmspe:.2f}")
print(f"Austin rank: {rank} out of {len(cities)}")
print(f"Placebo p-value: {p_value:.3f}")
print(f"{'✅ Statistically significant (p < 0.10)' if p_value < 0.10 else '❌ Not significant at 10% level'}")

## Step 7: What Happens When Pre-Fit Is Bad

A core assumption of synthetic control is that the **pre-treatment fit must be tight**. If we can't construct a synthetic version that tracks the treated unit before the intervention, we have no reason to trust the post-treatment gap. Below we demonstrate this by creating a city with an extreme trajectory that lies outside the convex hull of donors.

In [ ]:
extreme_records = []
extreme_revenue = 800 + 20 * months + np.random.normal(0, 10, n_months)
for i, m in enumerate(months):
    extreme_records.append({"city": "ExtremeCity", "month": m, "revenue": extreme_revenue[i]})

df_with_extreme = pd.concat([df, pd.DataFrame(extreme_records)], ignore_index=True)
all_cities_ext = cities + ["ExtremeCity"]

gap_extreme, pre_rmse_extreme = run_synthetic_control("ExtremeCity", df_with_extreme, all_cities_ext, treatment_month)

pivot_ext = df_with_extreme.pivot(index="month", columns="city", values="revenue")
donors_ext = [c for c in all_cities_ext if c != "ExtremeCity"]
pre_ext = pivot_ext.loc[:treatment_month, "ExtremeCity"].values
pre_donors_ext = pivot_ext.loc[:treatment_month, donors_ext].values

def obj_ext(w):
    return np.sum((pre_ext - pre_donors_ext @ w) ** 2)
n_d_ext = len(donors_ext)
res_ext = minimize(obj_ext, np.ones(n_d_ext) / n_d_ext, method="SLSQP",
                   bounds=[(0, 1)] * n_d_ext,
                   constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}])
synth_extreme = pivot_ext[donors_ext].values @ res_ext.x

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(months, pivot_ext["ExtremeCity"].values, label="ExtremeCity (actual)", linewidth=2)
axes[0].plot(months, synth_extreme, label="Synthetic ExtremeCity", linewidth=2, linestyle="--")
axes[0].axvline(x=treatment_month, color="red", linestyle=":", linewidth=1.5)
axes[0].set_title(f"Bad Pre-Fit (RMSE: ${pre_rmse_extreme:.1f}K)")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Revenue ($K)")
axes[0].legend()

axes[1].plot(months, actual_austin, label="Austin (actual)", linewidth=2)
axes[1].plot(months, synthetic_austin, label="Synthetic Austin", linewidth=2, linestyle="--")
axes[1].axvline(x=treatment_month, color="red", linestyle=":", linewidth=1.5)
axes[1].set_title(f"Good Pre-Fit (RMSE: ${placebo_pre_rmse['Austin']:.1f}K)")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Revenue ($K)")
axes[1].legend()

plt.suptitle("Pre-Treatment Fit Quality: Bad vs. Good", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"ExtremeCity pre-treatment RMSE: ${pre_rmse_extreme:.1f}K — POOR FIT")
print(f"Austin pre-treatment RMSE:      ${placebo_pre_rmse['Austin']:.1f}K — GOOD FIT")
print("\n⚠️  When pre-fit is poor, the post-treatment gap is NOT a credible causal estimate.")
print("The synthetic version can't reproduce pre-treatment outcomes, so we can't trust")
print("that any post-treatment divergence is due to treatment rather than model failure.")

## Key Takeaways

1. **Synthetic Control is designed for single-unit interventions** at an aggregate level (city, country, region) — exactly the setting where DiD, PSM, and A/B tests struggle.

2. **The method constructs a data-driven counterfactual** by finding the weighted combination of donor units that best reproduces the treated unit's pre-treatment trajectory.

3. **Pre-treatment fit is the key diagnostic.** If the synthetic control doesn't closely track the treated unit before the intervention, the post-treatment gap cannot be interpreted causally.

4. **Inference uses placebo (permutation) tests**, not traditional standard errors. The treated unit's effect is compared against placebo effects from all donor cities.

5. **Transparency of weights** is a major advantage — stakeholders can see exactly which cities contribute to the counterfactual and by how much.

6. **Limitations:** Requires the treated unit to be within the convex hull of donors. If the treated unit is too different from all donors, the method fails (as shown in the bad pre-fit example).